<img src="../assets/embedding_flow.png" alt="Embedding 유사도와 검색, 비즈니스 라우팅 흐름" style="display: block; width: 100%; max-width: 1200px; margin: 0 auto 24px; border-radius: 12px;">


## Embedding은 어떻게 문장을 비교할까요?

Embedding 모델은 문장을 의미를 담은 숫자 목록, 즉 **벡터**로 바꿉니다. 비슷한 의미의 문장은 벡터 공간에서 가까운 위치에 놓입니다.

<div style="display:flex; align-items:stretch; gap:10px; margin:18px 0; flex-wrap:wrap; font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;">
  <div style="flex:1 1 180px; padding:15px; border:1px solid #c7d2e1; border-radius:10px; background:#f8fafc;"><b>문장 A</b><br><span style="font-size:.9em; color:#4b5563;">카드에서 같은 금액이 두 번 결제됐어요.</span></div>
  <div style="display:flex; align-items:center; justify-content:center; color:#1a73e8; font-size:1.5em;">→</div>
  <div style="flex:1 1 145px; padding:15px; border:2px solid #8ab4f8; border-radius:10px; background:#eef6ff;"><b style="color:#1967d2;">Embedding 모델</b><br><code style="font-size:.82em;">[0.12, 0.87, …]</code></div>
  <div style="display:flex; align-items:center; justify-content:center; color:#1a73e8; font-size:1.5em;">↔</div>
  <div style="flex:1 1 180px; padding:15px; border:1px solid #c7d2e1; border-radius:10px; background:#f8fafc;"><b>문장 B</b><br><span style="font-size:.9em; color:#4b5563;">중복 결제된 주문을 확인해 주세요.</span></div>
  <div style="display:flex; align-items:center; justify-content:center; color:#188038; font-size:1.5em;">→</div>
  <div style="flex:1 1 230px; padding:15px; border:2px solid #b7dfc5; border-radius:10px; background:#f4fbf6;"><b style="color:#188038;">비교 함수</b><code style="display:block; margin-top:7px; font-size:.76em;">cosine_similarity(a, b)</code><code style="display:block; margin-top:4px; font-size:.76em;">euclidean_distance(a, b)</code><code style="display:block; margin-top:4px; font-size:.76em;">manhattan_distance(a, b)</code></div>
</div>

| 함수 | 무엇을 비교하나 | 특징 | 그림 |
| --- | --- | --- | --- |
| cosine_similarity | 두 벡터가 향하는 방향 | 방향 중심 · 텍스트 검색 기본 | <span style="white-space:nowrap; color:#1a73e8; font-size:1.15em;">● ↗　● ↗</span><br><small>같은 방향</small> |
| euclidean_distance | 두 벡터 사이의 직선 거리 | 방향 + 벡터 크기 반영 | <span style="white-space:nowrap; color:#9334e6; font-size:1.15em;">● ───── ●</span><br><small>직선 경로</small> |
| manhattan_distance | 각 숫자 차이를 모두 더한 거리 | 차원별 차이 합산 · 이상치 영향 완화 | <span style="display:inline-block; color:#e37400; font-size:1.05em; line-height:1.05;">● ──┐<br>　　└── ●</span><br><small>격자 경로</small> |

텍스트 embedding 검색은 보통 문장의 길이보다 의미 방향에 초점을 맞추기 위해 cosine similarity를 기본으로 사용합니다. 벡터를 정규화하면 cosine과 Euclidean의 순위가 매우 비슷해질 수 있지만, Manhattan은 다른 순위를 낼 수 있습니다.

그래서 단어가 완전히 같지 않아도 의미가 가까운 문장과 문서를 찾을 수 있습니다. 이 차트는 많은 차원의 벡터를 사람이 보기 쉽게 2차원으로 투영한 그림입니다.

> **Embedding은 문장을 각각 벡터로 바꾼 뒤, 벡터 사이의 거리를 비교해 비슷한 문장을 찾습니다.**


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124;">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2;">📘 실습 1 · 문장 유사도와 2D 클러스터 (UI)</div>
  <p style="margin: 8px 0 0; line-height: 1.6;">두 문장을 비교해 cosine similarity를 확인하고, 누적된 문장을 2D 지도에서 살펴봅니다. 새 항목을 추가해도 기존 점의 좌표는 유지됩니다.</p>
  <p style="margin: 7px 0 0; line-height: 1.6;">UI는 임베딩 요청만 수행하며 외부 업무 상태를 변경하지 않습니다.</p>
</div>

In [1]:
from pathlib import Path
import importlib
import sys

project_root = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / '1_gen_ai_hub' / 'src').exists()
)
for module_path in (project_root / '1_gen_ai_hub' / 'src', project_root / '2_gen_ai_hub_llm'):
    if str(module_path) not in sys.path:
        sys.path.insert(0, str(module_path))

import src.notebook_ui as notebook_ui
import src.embeddings.lab_core as embedding_core
import src.business_embedding.lab_core as business_embedding_core
from src.embeddings.lesson_support.config import load_settings
from src.embeddings.lab_core import compare_embeddings, retrieve_similar_texts
from src.business_embedding.lab_core import load_business_cases, run_business_embedding_lab

importlib.reload(notebook_ui)
importlib.reload(embedding_core)
importlib.reload(business_embedding_core)

settings = load_settings()

In [2]:
embedding_ui = notebook_ui.build_embedding_ui(
    embedding_core.compare_and_append_embedding_clusters
)
embedding_ui.launch(
    inline=True, share=False, show_error=True, server_name='127.0.0.1',
    width='80%', height=1000,
)

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124;">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2;">📘 실습 2 · 비즈니스 문의 분석과 추천 큐 (UI)</div>
  <p style="margin: 8px 0 0; line-height: 1.6;">합성된 과거 사례를 기준으로 새 문의의 cluster, 추천 큐, 라벨별 유사도를 확인합니다. 낮은 유사도·작은 margin·신규 유형은 자동 배정 대신 사람 검토 대상으로 표시됩니다.</p>
  <p style="margin: 7px 0 0; line-height: 1.6;">UI는 분석과 추천만 수행하며 실제 업무 큐나 외부 상태를 변경하지 않습니다.</p>
</div>

In [3]:
if 'business_embedding_ui' in globals():
    business_embedding_ui.close()

business_embedding_ui = notebook_ui.build_business_embedding_ui(
    run_business_embedding_lab,
    cases=load_business_cases(),
    n_clusters=4,
)
business_embedding_ui.launch(
    inline=True, share=False, show_error=True, server_name='127.0.0.1',
    width='100%', height=520,
)

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


<div style="margin: 32px 0 12px; padding: 13px 15px; border-left: 4px solid #5f6368; background: #f1f3f4; border-radius: 6px; color: #202124;">
  <div style="font-size: .82em; font-weight: 700; color: #5f6368;">🧩 코드 베이스 실습</div>
  <h2 style="margin: 8px 0; font-size: 1.05em;">3. 문장 유사도를 코드로 확인하기</h2>
  <p style="margin: 0; line-height: 1.6;">UI가 호출하는 코어 API로 여러 문장의 cosine similarity를 한 번에 계산합니다. 단어가 비슷하거나 반대 의미인 문장의 점수도 함께 관찰하세요.</p>
</div>

In [4]:
examples = [
    '결제를 취소하고 싶어요.',
    '주문 환불을 요청합니다.',
    '카드 배송 상태를 알려주세요.',
    '상품 배송 상태를 알려주세요.',
    '환불은 원하지 않아요.',
]
comparison = compare_embeddings(examples, settings)
print(f"embedding model: {comparison['model']}")
print(f"vector dimensions: {comparison['dimensions']}")
for item in comparison['comparisons']:
    print(f"{item['left']} ↔ {item['right']}: {item['cosine_similarity']:.3f}")

embedding model: text-embedding-3-large
vector dimensions: 3072
결제를 취소하고 싶어요. ↔ 주문 환불을 요청합니다.: 0.535
결제를 취소하고 싶어요. ↔ 카드 배송 상태를 알려주세요.: 0.323
결제를 취소하고 싶어요. ↔ 상품 배송 상태를 알려주세요.: 0.263
결제를 취소하고 싶어요. ↔ 환불은 원하지 않아요.: 0.482
주문 환불을 요청합니다. ↔ 카드 배송 상태를 알려주세요.: 0.400
주문 환불을 요청합니다. ↔ 상품 배송 상태를 알려주세요.: 0.417
주문 환불을 요청합니다. ↔ 환불은 원하지 않아요.: 0.560
카드 배송 상태를 알려주세요. ↔ 상품 배송 상태를 알려주세요.: 0.775
카드 배송 상태를 알려주세요. ↔ 환불은 원하지 않아요.: 0.238
상품 배송 상태를 알려주세요. ↔ 환불은 원하지 않아요.: 0.232


In [5]:
query = '결제는 취소했는데 환불금이 아직 안 들어왔어요.'
faq_documents = [
    '결제 취소는 주문 상세 화면에서 요청할 수 있습니다.',
    '환불 완료 후 카드사 반영까지 영업일 기준 3~5일이 걸릴 수 있습니다.',
    '배송지를 변경하려면 상품 준비 전 고객센터에 문의하세요.',
    '카드 배송 조회는 카드사 앱에서 확인할 수 있습니다.',
    '중복 결제된 주문은 결제 내역과 주문 번호를 함께 보내주세요.',
]
retrieval = retrieve_similar_texts(query, faq_documents, settings, top_k=3)
for item in retrieval['results']:
    print(f"{item['rank']}. ({item['cosine_similarity']:.3f}) {item['document']}")

threshold = 0.55
filtered = retrieve_similar_texts(query, faq_documents, settings, top_k=3, min_similarity=threshold)
print(f'\n{threshold} 이상: {len(filtered["results"])}건')

1. (0.550) 결제 취소는 주문 상세 화면에서 요청할 수 있습니다.
2. (0.504) 환불 완료 후 카드사 반영까지 영업일 기준 3~5일이 걸릴 수 있습니다.
3. (0.356) 중복 결제된 주문은 결제 내역과 주문 번호를 함께 보내주세요.

0.55 이상: 0건


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124;">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2;">📘 실습 4 · 검색 정책 관찰</div>
  <h2 style="margin: 8px 0; font-size: 1.05em;">threshold와 chunk 크기 비교</h2>
  <p style="margin: 0; line-height: 1.6;">threshold는 관련 없는 문서를 제외하는 기준입니다. 짧은 chunk와 긴 문서의 Top-1 점수를 비교해, 문서 분할 방식이 검색 품질에 주는 영향을 확인하세요.</p>
</div>

In [6]:
chunk_query = '환불금은 언제 카드에 반영되나요?'
short_chunks = [
    '환불 완료 후 카드사 반영까지 영업일 기준 3~5일이 걸릴 수 있습니다.',
    '계좌 환불은 등록된 계좌 정보를 확인한 뒤 처리합니다.',
    '배송 중인 상품은 취소가 제한될 수 있습니다.',
]
long_chunk = ['환불 완료 후 카드사 반영까지 영업일 기준 3~5일이 걸릴 수 있습니다. 계좌 환불은 등록된 계좌 정보를 확인한 뒤 처리합니다. 배송 중인 상품은 취소가 제한될 수 있습니다.']
for label, chunks in [('짧은 chunk', short_chunks), ('긴 문서', long_chunk)]:
    result = retrieve_similar_texts(chunk_query, chunks, settings, top_k=1)
    best = result['results'][0]
    print(f'{label}: {best["cosine_similarity"]:.3f} | {best["document"]}')

짧은 chunk: 0.746 | 환불 완료 후 카드사 반영까지 영업일 기준 3~5일이 걸릴 수 있습니다.
긴 문서: 0.655 | 환불 완료 후 카드사 반영까지 영업일 기준 3~5일이 걸릴 수 있습니다. 계좌 환불은 등록된 계좌 정보를 확인한 뒤 처리합니다. 배송 중인 상품은 취소가 제한될 수 있습니다.


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124;">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2;">📘 실습 5 · 비즈니스 라우팅 근거 읽기</div>
  <p style="margin: 8px 0 0; line-height: 1.6;">cluster ID 자체에는 업무 의미가 없습니다. 추천 큐뿐 아니라 라벨별 유사도와 holdout 결과를 함께 검토하세요.</p>
  <p style="margin: 7px 0 0; line-height: 1.6;">운영에서는 개인정보를 제거한 사례와 사람이 확정한 큐 라벨을 사용하고, 낮은 신뢰도는 사람 검토로 보냅니다.</p>
</div>

In [7]:
cases = load_business_cases()
new_ticket = '운송장에는 배송 완료로 나오지만 고객이 물건을 받지 못했습니다.'
outcome = run_business_embedding_lab(
    ticket=new_ticket, cases=cases, settings=settings, n_clusters=4,
)
print(f"embedding model: {outcome['model']}")
print(f"case count: {outcome['case_count']} | dimensions: {outcome['embedding_dimensions']}")
print(f"cluster inertia: {outcome['cluster_inertia']:.4f}")
print(f"holdout accuracy: {outcome['holdout']['accuracy']:.3f}")
print('추천 큐:', outcome['recommendation']['recommended_queue'])
print('라벨별 유사도:', outcome['recommendation']['label_similarity'])
print(outcome['action'])

embedding model: text-embedding-3-large
case count: 12 | dimensions: 3072
cluster inertia: 4.5980
holdout accuracy: 0.500
추천 큐: logistics
라벨별 유사도: {'logistics': 0.44921175937382385, 'technical': 0.3730724189851362, 'payment': 0.11579062046908706, 'compliance': 0.08815938046275319}
추천과 평가만 생성했습니다. 외부 업무 큐나 상태는 변경하지 않았습니다.


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1a73e8; background: #e8f0fe; border-radius: 6px; color: #202124;">
  <div style="font-size: .82em; font-weight: 700; color: #1a73e8;">✅ 미니 과제</div>
  <ol style="margin: 8px 0 0; padding-left: 1.3em; line-height: 1.7;"><li>한 업무 도메인의 FAQ를 5개 이상 작성하고, 두 질문의 Top-3와 threshold를 확인합니다.</li><li>새 업무 문의를 바꿔가며 추천 큐, 최고 유사도, 사람 검토 여부를 관찰합니다.</li><li>실제 자동 배정은 이 노트북 밖에서 권한 검증과 감사 로그를 거쳐서만 수행합니다.</li></ol>
</div>